In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr


In [ ]:
# ds = xr.open_zarr('/mnt/tier2/project/p200177/u101329/DE371_bis/diffusion-interpolation/_work/14-latent-edm-intrp/14-edm-z64-20260609_223027/samples.zarr', zarr_format=3, consolidated=True)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec


def _resolve_indices(coord_values, selection):
    """
    Always treat ``selection`` as *positional* indices (supports negative
    indexing like -1 for last).  Returns a plain Python list of coordinate
    *values* suitable for xarray .sel().

    Examples
    --------
    coord_values = [0, 5, 10, 15]
    _resolve_indices(coord_values, None)     → [0, 5, 10, 15]
    _resolve_indices(coord_values, [-1])     → [15]
    _resolve_indices(coord_values, [0, -1])  → [0, 15]
    _resolve_indices(coord_values, [1, 2])   → [5, 10]
    """
    arr = np.asarray(coord_values)
    if selection is None:
        return arr.tolist()
    return [arr[int(i)].item() for i in selection]


def plot_generated_vs_truth(ds, epochs=None, steps=None, variables=None, samples=None):
    """
    Plot generated vs ground-truth reconstructions in a 3-row layout.

    Layout per (epoch, step, variable) figure
    ──────────────────────────────────────────
    Row 0  │ Generated      │ col per sample
    Row 1  │ Ground Truth   │ col per sample
    Row 2  │ Difference     │ col per sample

    Parameters
    ----------
    ds        : xarray.Dataset with dims (epoch, sample, step, variable, ensemble, y, x)
                and data variables  ``generated``  and  ``ground_truth``.
    epochs    : positional indices into ds.epoch   (e.g. [-1] = last epoch).  None → all.
    steps     : positional indices into ds.step    (e.g. [0, -1] = first & last). None → all.
    variables : positional indices into ds.variable (e.g. [0] = first var).   None → all.
    samples   : positional indices into ds.sample  (e.g. [0,1,2,3]).          None → all.

    All index arguments support negative indexing (Python-style).
    """

    # ── 0. Resolve positional indices → coordinate values ───────────────────
    epoch_list  = _resolve_indices(ds.epoch.values,    epochs)
    step_list   = _resolve_indices(ds.step.values,     steps)
    var_list    = _resolve_indices(ds.variable.values, variables)
    sample_list = _resolve_indices(ds.sample.values,   samples)

    n_samples = len(sample_list)

    # ── 1. Slice first, then compute ─────────────────────────────────────────
    print("Loading data subset…")
    ds_sub = ds.sel(epoch=epoch_list, sample=sample_list,
                    step=step_list, variable=var_list)

    # squeeze ensemble & time dims → shape: (epoch, sample, step, variable, y, x)
    gen_all = ds_sub.generated   .squeeze("ensemble").compute()
    grt_all = ds_sub.ground_truth.squeeze("ensemble").compute()

    gen_np  = gen_all.values.astype(np.float32)   # (E, S, T, V, y, x)
    grt_np  = grt_all.values.astype(np.float32)

    # Index maps into subset arrays
    epoch_idx = {e: i for i, e in enumerate(epoch_list)}
    samp_idx  = {s: i for i, s in enumerate(sample_list)}
    step_idx  = {t: i for i, t in enumerate(step_list)}
    var_idx   = {v: i for i, v in enumerate(var_list)}

    # ── 2. Colour scales per variable ────────────────────────────────────────
    print("Computing colour scales…")
    var_scales  = {}   # (vmin, vmax) shared across gen & grt
    diff_scales = {}   # symmetric (−d, +d)
    for var in var_list:
        vi = var_idx[var]
        g  = gen_np[:, :, :, vi]
        r  = grt_np[:, :, :, vi]
        var_scales[var]  = (float(min(g.min(), r.min())),
                            float(max(g.max(), r.max())))
        diff_scales[var] = float(np.abs(g - r).max())

    step_labels = {s: ds.attrs.get(f"step_{s}", f"step {s}") for s in step_list}

    # ── 3. Plot loop ─────────────────────────────────────────────────────────
    print("Plotting…")
    row_labels = ["Generated", "Ground Truth", "Difference"]

    for epoch in epoch_list:
        ei = epoch_idx[epoch]

        for step in step_list:
            ti = step_idx[step]
            step_label = step_labels[step]

            for var in var_list:
                vi         = var_idx[var]
                vmin, vmax = var_scales[var]
                d_abs      = diff_scales[var]
                cmap_main  = "Blues" if var == "tp" else "magma"

                # ── figure & gridspec ────────────────────────────────────────
                fig = plt.figure(figsize=(4.5 * n_samples + 1.5, 13),
                                 constrained_layout=False)
                fig.suptitle(
                    f"Epoch {epoch}  │  {var}  │  {step_label}",
                    fontsize=14, fontweight="bold", y=0.98,
                )

                # 3 rows × n_samples cols + 1 narrow col for colorbars
                gs = gridspec.GridSpec(
                    3, n_samples + 1,
                    figure=fig,
                    width_ratios=[1] * n_samples + [0.06],
                    hspace=0.35, wspace=0.04,
                    left=0.06, right=0.94, top=0.93, bottom=0.04,
                )

                axes = np.empty((3, n_samples), dtype=object)
                for r in range(3):
                    for c in range(n_samples):
                        axes[r, c] = fig.add_subplot(gs[r, c])

                cb_ax_main = fig.add_subplot(gs[0:2, n_samples])
                cb_ax_diff = fig.add_subplot(gs[2,   n_samples])

                im_main = im_diff = None

                for s_idx, sample in enumerate(sample_list):
                    si   = samp_idx[sample]
                    gen  = np.flipud(gen_np[ei, si, ti, vi].astype(np.float64))
                    grt  = np.flipud(grt_np[ei, si, ti, vi].astype(np.float64))
                    diff = gen - grt
                    rmse = float(np.sqrt(np.mean(diff ** 2)))

                    diff_min = float(diff.min())
                    diff_max = float(diff.max())
                    min_loc  = np.unravel_index(np.argmin(diff), diff.shape)
                    max_loc  = np.unravel_index(np.argmax(diff), diff.shape)

                    col_title = f"Sample {sample}"

                    # ── Row 0 : Generated ────────────────────────────────────
                    ax = axes[0, s_idx]
                    im_main = ax.imshow(gen, cmap=cmap_main, vmin=vmin, vmax=vmax)
                    ax.set_title(col_title if s_idx == n_samples // 2 else col_title,
                                 fontsize=9)
                    ax.axis("off")
                    if s_idx == 0:
                        ax.set_ylabel(row_labels[0], fontsize=9, labelpad=4)
                        ax.yaxis.set_visible(True)
                        ax.tick_params(left=False, labelleft=False)

                    # ── Row 1 : Ground Truth ─────────────────────────────────
                    ax = axes[1, s_idx]
                    ax.imshow(grt, cmap=cmap_main, vmin=vmin, vmax=vmax)
                    ax.set_title("", fontsize=9)
                    ax.axis("off")
                    if s_idx == 0:
                        ax.set_ylabel(row_labels[1], fontsize=9, labelpad=4)
                        ax.yaxis.set_visible(True)
                        ax.tick_params(left=False, labelleft=False)

                    # ── Row 2 : Difference ───────────────────────────────────
                    ax = axes[2, s_idx]
                    im_diff = ax.imshow(diff, cmap="RdBu_r", vmin=-d_abs, vmax=d_abs)
                    ax.set_title(f"RMSE: {rmse:.4f}", fontsize=8)
                    ax.axis("off")
                    if s_idx == 0:
                        ax.set_ylabel(row_labels[2], fontsize=9, labelpad=4)
                        ax.yaxis.set_visible(True)
                        ax.tick_params(left=False, labelleft=False)

                    # Min / max markers on difference panel
                    for loc, marker, color, label, offset in [
                        (min_loc, "v", "blue", f"Min {diff_min:.3f}", ( 6,   6)),
                        (max_loc, "^", "red",  f"Max {diff_max:.3f}", ( 6, -14)),
                    ]:
                        axes[2, s_idx].plot(
                            loc[1], loc[0], marker, color=color,
                            markersize=7, markeredgecolor="white",
                            markeredgewidth=0.8,
                        )
                        axes[2, s_idx].annotate(
                            label.replace(" ", "\n"),
                            xy=(loc[1], loc[0]), xytext=offset,
                            textcoords="offset points",
                            color=color, fontsize=7, fontweight="bold",
                            bbox=dict(boxstyle="round,pad=0.2",
                                      fc="white", alpha=0.65, ec="none"),
                        )

                    # Column titles (only row 0 gets them)
                    axes[0, s_idx].set_title(f"Sample {sample}", fontsize=9)

                # ── Row labels on left edge ──────────────────────────────────
                for r_idx, label in enumerate(row_labels):
                    axes[r_idx, 0].text(
                        -0.12, 0.5, label,
                        transform=axes[r_idx, 0].transAxes,
                        fontsize=10, fontweight="bold", va="center",
                        ha="right", rotation=90, color="#333333",
                    )

                # ── Colorbars ────────────────────────────────────────────────
                fig.colorbar(im_main, cax=cb_ax_main, orientation="vertical"
                             ).set_label(f"{var}  (gen / grt)", fontsize=8)
                fig.colorbar(im_diff, cax=cb_ax_diff, orientation="vertical"
                             ).set_label(f"{var}  difference", fontsize=8)

                plt.show()

In [ ]:


file_paths = {
    #'14-latent-edm-intrp/14-edm-z32':'/mnt/tier2/project/p200177/u101329/DE371_bis/diffusion-interpolation/_work/14-latent-edm-intrp/14-edm-z32-20260610_015044/samples.zarr',  
    #'14-latent-edm-intrp/14-edm-z64':'/mnt/tier2/project/p200177/u101329/DE371_bis/diffusion-interpolation/_work/14-latent-edm-intrp/14-edm-z64-20260610_015050/samples.zarr',
    #'14-edm-z32-20260612_003244':'/mnt/tier2/project/p200177/u101329/DE371_bis/diffusion-interpolation/_work/14-latent-edm-intrp/14-edm-z32-20260612_003244/samples.zarr',
    #'14-edm-z64-20260612_003248':'/mnt/tier2/project/p200177/u101329/DE371_bis/diffusion-interpolation/_work/14-latent-edm-intrp/14-edm-z64-20260612_003248/samples.zarr',

'18-lnd-z32-qrl-vae-20260612_174451':'/mnt/tier2/project/p200177/u101329/DE371_bis/diffusion-interpolation/_work/18-lnd-interp/18-lnd-z32-qrl-vae-20260612_174451/samples.zarr',
'18-lnd-z32-qrl-vae-20260612_203416':'/mnt/tier2/project/p200177/u101329/DE371_bis/diffusion-interpolation/_work/18-lnd-interp/18-lnd-z32-qrl-vae-20260612_203416/samples.zarr',
#'18-lnd-z64-lola-20260612_173622':'/mnt/tier2/project/p200177/u101329/DE371_bis/diffusion-interpolation/_work/18-lnd-interp/18-lnd-z64-lola-20260612_173622/samples.zarr',
#'18-lnd-z32-20260612_173621':'/mnt/tier2/project/p200177/u101329/DE371_bis/diffusion-interpolation/_work/18-lnd-interp/18-lnd-z32-20260612_173621/samples.zarr',

}



In [ ]:
for label, file_path in file_paths.items():
    print(f"\n{'='*60}")
    print(f"  {label}  —  {file_path.split('/')[-2]}")
    print(f"{'='*60}")
    try:
        ds = xr.open_zarr(file_path, zarr_format=3, consolidated=True)
        plot_generated_vs_truth(ds, steps=[3], epochs=[-1], variables=[0])
        print(ds)
    except Exception as e:
        print(f"  ERROR: {e}")

In [ ]:
for label, file_path in file_paths.items():
    print(f"\n{'='*60}")
    print(f"  {label}  —  {file_path.split('/')[-2]}")
    print(f"{'='*60}")
    try:
        ds = xr.open_zarr(file_path, zarr_format=3, consolidated=True)
        print(ds)
        plot_generated_vs_truth(ds, steps=[3],epochs=[-1], variables=[1])
    except Exception as e:
        print(f"  ERROR: {e}")

In [ ]:
for label, file_path in file_paths.items():
    print(f"\n{'='*60}")
    print(f"  {label}  —  {file_path.split('/')[-2]}")
    print(f"{'='*60}")
    try:
        ds = xr.open_zarr(file_path, zarr_format=3, consolidated=True)
        print(ds)
        plot_generated_vs_truth(ds, steps=[3], epochs=[-1], variables=[2])
    except Exception as e:
        print(f"  ERROR: {e}")

In [ ]:
for label, file_path in file_paths.items():
    print(f"\n{'='*60}")
    print(f"  {label}  —  {file_path.split('/')[-2]}")
    print(f"{'='*60}")
    try:
        ds = xr.open_zarr(file_path, zarr_format=3, consolidated=True)
        print(ds)
        plot_generated_vs_truth(ds, steps=[3], epochs=[-1], variables=[3])
    except Exception as e:
        print(f"  ERROR: {e}")

In [ ]:
for label, file_path in file_paths.items():
    print(f"\n{'='*60}")
    print(f"  {label}  —  {file_path.split('/')[-2]}")
    print(f"{'='*60}")
    try:
        ds = xr.open_zarr(file_path, zarr_format=3, consolidated=True)
        plot_generated_vs_truth(ds, epochs=[-1], variables=[4])
    except Exception as e:
        print(f"  ERROR: {e}")

In [ ]:
for label, file_path in file_paths.items():
    print(f"\n{'='*60}")
    print(f"  {label}  —  {file_path.split('/')[-2]}")
    print(f"{'='*60}")
    try:
        ds = xr.open_zarr(file_path, zarr_format=3, consolidated=True)
        print(ds)
        plot_generated_vs_truth(ds, steps=[3], epochs=[-1], variables=[5])
    except Exception as e:
        print(f"  ERROR: {e}")